In [0]:
%run "../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Create catalogs
for catalog in CATALOGS.values():
    catalog_exists = spark.catalog.databaseExists(catalog)
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    status = "Already existed" if catalog_exists else "Created successfully"
    print(f"{catalog}: {status}")

In [0]:
# Create schemas
schema_mapping = {
    "bronze": [SCHEMAS["bronze"]],
    "silver": [SCHEMAS["staging"], SCHEMAS["intermediate"]],
    "gold": ["intermediate", "marts", "reporting"]
}

for catalog_key, schema_list in schema_mapping.items():
    catalog = CATALOGS[catalog_key]

    existing_schemas = {
        list(row.asDict().values())[0]
        for row in spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()
    }

    for schema in schema_list:
        schema_name = f"{catalog}.{schema}"
        schema_exists = schema in existing_schemas

        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

        status = "Already existed" if schema_exists else "Created successfully"

        print(f"{schema_name}: {status}")

In [0]:
# Create Bronze volume
volume_exists = spark.sql(
    f"SHOW VOLUMES IN {CATALOGS['bronze']}.{SCHEMAS['bronze']}"
).filter(
    F.col("volume_name") == "raw"
).count() > 0

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {RAW_VOLUME}"
)

status = "Already existed" if volume_exists else "Created successfully"

print(f"{RAW_VOLUME}: {status}")

In [0]:
# Validate infrastructure
print("======================================")
print("ENVIRONMENT VALIDATION")
print("======================================")

# Validate catalogs
existing_catalogs = {
    list(row.asDict().values())[0]
    for row in spark.sql("SHOW CATALOGS").collect()
}

for catalog in CATALOGS.values():
    if catalog not in existing_catalogs:
        raise ValueError(f"Catalog not found: {catalog}")

    print(f"Catalog validated: {catalog}")

# Validate schemas
for catalog_key, schema_list in schema_mapping.items():
    catalog = CATALOGS[catalog_key]

    existing_schemas = {
        list(row.asDict().values())[0]
        for row in spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()
    }

    for schema in schema_list:
        if schema not in existing_schemas:
            raise ValueError(f"Schema not found: {catalog}.{schema}")

        print(f"Schema validated: {catalog}.{schema}")

# Validate Bronze volume
existing_volumes = {
    row["volume_name"]
    for row in spark.sql(
        f"SHOW VOLUMES IN {CATALOGS['bronze']}.{SCHEMAS['bronze']}"
    ).collect()
}

if "raw" not in existing_volumes:
    raise ValueError(f"Volume not found: {RAW_VOLUME}")

print(f"Volume validated: {RAW_VOLUME}")

# Validate volume accessibility
volume_contents = dbutils.fs.ls(
    f"/Volumes/{CATALOGS['bronze']}/{SCHEMAS['bronze']}/raw/"
)

print(f"Volume accessible: {len(volume_contents)} entries")

print("======================================")
print("Infrastructure setup completed successfully.")
print("======================================")